# MediScan — Day 6: Training Loop Setup

## Objective

The objective of Day 6 is to configure the training components
required for EfficientNet-B0 training.

This includes:

- CrossEntropyLoss
- Adam optimizer
- Learning rate scheduler
- Training and validation step structure

Actual multi-epoch training will be performed in Day 7.

In [2]:
# ============================================
# Restore Day 5 Model for Day 6
# ============================================

import torch
import torch.nn as nn
import torchvision
from torchvision import models

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Number of classes
NUM_CLASSES = 4

# Load pre-trained EfficientNet-B0
weights = models.EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(
    weights=weights
)

# Replace classifier
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

# Freeze base model
for param in model.features.parameters():
    param.requires_grad = False

# Move model to device
model = model.to(device)

print("Day 5 model restored successfully!")
print("Architecture: EfficientNet-B0")
print("Device:", device)
print("Classes:", NUM_CLASSES)
print("Classifier:", model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 197MB/s]


Day 5 model restored successfully!
Architecture: EfficientNet-B0
Device: cuda
Classes: 4
Classifier: Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


In [3]:
# ============================================
# Day 6 - Loss Function
# ============================================

criterion = nn.CrossEntropyLoss()

print("Loss function:")
print(criterion)

Loss function:
CrossEntropyLoss()


In [4]:
# ============================================
# Day 6 - Adam Optimizer
# ============================================

LEARNING_RATE = 1e-3

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE
)

print("Optimizer:")
print(optimizer)

print("\nLearning rate:", LEARNING_RATE)

Optimizer:
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

Learning rate: 0.001


In [5]:
# ============================================
# Day 6 - Learning Rate Scheduler
# ============================================

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)

print("Learning rate scheduler:")
print(scheduler)

print("\nStep size:", scheduler.step_size)
print("Gamma:", scheduler.gamma)

Learning rate scheduler:

Step size: 5
Gamma: 0.1


In [12]:
# ============================================
# Restore Google Drive
# ============================================

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os

print("MyDrive exists:",
      os.path.exists('/content/drive/MyDrive'))

print("\nFolders inside MyDrive:")

for item in os.listdir('/content/drive/MyDrive'):
    print("📁/📄", item)

MyDrive exists: True

Folders inside MyDrive:
📁/📄 New Folder
📁/📄 Sagar.jpg
📁/📄 IMG-20250906-WA0002.jpg
📁/📄 IMG-20251110-WA0001.jpg
📁/📄 OKIE DOKIE APP
📁/📄 Knowledge Base
📁/📄 IMG_20260330_181523.png
📁/📄 Nishu Bhabhi
📁/📄 Colab Notebooks
📁/📄 IMG-20260527-WA0003.jpg
📁/📄 IMG-20260527-WA0002.jpg
📁/📄 IMG-20260527-WA0001.jpg
📁/📄 project ppt 
📁/📄 DATA ANALYST INTERNSHIP 
📁/📄 ey1.pdf
📁/📄 ey2.pdf
📁/📄 IMG_20260623_095508.jpg
📁/📄 Resume_Enhanced.pdf
📁/📄 ibm_offer_letter.pdf
📁/📄 IMG_20260709_084613.jpg
📁/📄 IMG_20260709_084627.jpg
📁/📄 IMG_20260709_084720.jpg
📁/📄 IBM_Certificate.pdf
📁/📄 Ticket_Categorizer_Submission.pdf
📁/📄 Sagar_Sharma_Resume.pdf
📁/📄 archive.zip
📁/📄 MediScan


In [14]:
# ============================================
# Locate MediScan Project
# ============================================

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/MediScan")

print("Project exists:", PROJECT_DIR.exists())
print("Project path:", PROJECT_DIR)

print("\nContents of MediScan:")

for item in PROJECT_DIR.iterdir():
    print("📁/📄", item.name)

Project exists: True
Project path: /content/drive/MyDrive/MediScan

Contents of MediScan:
📁/📄 data
📁/📄 figures
📁/📄 reports


In [15]:
# ============================================
# Check Processed Data
# ============================================

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Data directory exists:", DATA_DIR.exists())
print("Processed directory exists:", PROCESSED_DIR.exists())

if DATA_DIR.exists():
    print("\nContents of data:")
    for item in DATA_DIR.iterdir():
        print("📁/📄", item.name)

if PROCESSED_DIR.exists():
    print("\nContents of processed:")
    for item in PROCESSED_DIR.iterdir():
        print("📄", item.name)

Data directory exists: True
Processed directory exists: True

Contents of data:
📁/📄 processed
📁/📄 raw

Contents of processed:
📄 train.csv
📄 val.csv
📄 test.csv
📄 class_to_idx.json


In [16]:
# ============================================
# Day 6 - Restore Existing Dataset CSVs
# ============================================

import pandas as pd

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nColumns:")
print(train_df.columns.tolist())

Train: 5040
Validation: 1080
Test: 1080

Columns:
['image_path', 'label', 'source']


In [17]:
# ============================================
# Day 6 - Restore Dataset Pipeline
# ============================================

from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


train_augmentation = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.1,
        hue=0.02
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05),
        shear=5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


evaluation_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class BrainMRIDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["image_path"]
        label = int(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


print("Dataset pipeline restored successfully!")

Dataset pipeline restored successfully!


In [18]:
# ============================================
# Day 6 - Restore Dataset Objects
# ============================================

train_dataset = BrainMRIDataset(
    train_df,
    transform=train_augmentation
)

val_dataset = BrainMRIDataset(
    val_df,
    transform=evaluation_transform
)

test_dataset = BrainMRIDataset(
    test_df,
    transform=evaluation_transform
)

print("Datasets restored successfully!")
print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Datasets restored successfully!
Train dataset: 5040
Validation dataset: 1080
Test dataset: 1080


In [19]:
# ============================================
# Day 6 - Restore Efficient DataLoaders
# ============================================

from torch.utils.data import DataLoader

BATCH_SIZE = 32
NUM_WORKERS = 2
PREFETCH_FACTOR = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=PREFETCH_FACTOR
)

print("DataLoaders restored successfully!")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders restored successfully!
Train batches: 158
Validation batches: 34
Test batches: 34


In [21]:
# ============================================
# Check Existing Class Mapping
# ============================================

import json

with open(PROCESSED_DIR / "class_to_idx.json", "r") as f:
    class_to_idx = json.load(f)

print("Class mapping:")
print(class_to_idx)

Class mapping:
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [22]:
# ============================================
# Day 6 - Corrected BrainMRIDataset
# ============================================

class BrainMRIDataset(Dataset):

    def __init__(self, dataframe, transform=None, class_to_idx=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["image_path"]
        class_name = row["label"]

        # Convert class name → integer index
        label = self.class_to_idx[class_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


print("Corrected BrainMRIDataset ready!")

Corrected BrainMRIDataset ready!


In [23]:
# ============================================
# Day 6 - Recreate Dataset Objects
# ============================================

train_dataset = BrainMRIDataset(
    train_df,
    transform=train_augmentation,
    class_to_idx=class_to_idx
)

val_dataset = BrainMRIDataset(
    val_df,
    transform=evaluation_transform,
    class_to_idx=class_to_idx
)

test_dataset = BrainMRIDataset(
    test_df,
    transform=evaluation_transform,
    class_to_idx=class_to_idx
)

print("Datasets recreated successfully!")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Datasets recreated successfully!
Train: 5040
Validation: 1080
Test: 1080


In [25]:
# ============================================
# Day 6 - Recreate DataLoaders
# ============================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

print("DataLoaders recreated successfully!")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders recreated successfully!
Train batches: 158
Validation batches: 34
Test batches: 34


In [26]:
# ============================================
# Day 6 - Loss Compatibility Test
# ============================================

model.eval()

with torch.no_grad():
    images, labels = next(iter(train_loader))

    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)
print("Labels dtype:", labels.dtype)
print("Loss value  :", loss.item())

FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_872/2235618365.py", line 25, in __getitem__
    image = Image.open(image_path).convert("RGB")
            ~~~~~~~~~~^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
FileNotFoundError: [Errno 2] No such file or directory: '/content/MediScan/Training/pituitary/Tr-pi_5.jpg'


In [27]:
# ============================================
# Check Actual Dataset Path
# ============================================

from pathlib import Path

RAW_DIR = PROJECT_DIR / "data" / "raw"

print("Raw directory:", RAW_DIR)
print("Exists:", RAW_DIR.exists())

actual_image = RAW_DIR / "Training" / "pituitary" / "Tr-pi_5.jpg"

print("Image path:", actual_image)
print("Image exists:", actual_image.exists())

Raw directory: /content/drive/MyDrive/MediScan/data/raw
Exists: True
Image path: /content/drive/MyDrive/MediScan/data/raw/Training/pituitary/Tr-pi_5.jpg
Image exists: True


In [28]:
# ============================================
# Day 6 - Test Dataset Image
# ============================================

image, label = train_dataset[0]

print("Image shape :", image.shape)
print("Image dtype :", image.dtype)
print("Label       :", label)
print("Label type  :", type(label))

FileNotFoundError: [Errno 2] No such file or directory: '/content/MediScan/Training/notumor/Tr-no_626.jpg'

In [29]:
# ============================================
# Day 6 - Robust Image Path Resolver
# ============================================

from pathlib import Path

RAW_DIR = PROJECT_DIR / "data" / "raw"


def resolve_image_path(csv_path):
    """
    Resolve image path from the existing CSV to the
    actual image stored inside Google Drive.
    """

    csv_path = Path(str(csv_path))

    # 1. If the path already exists, use it
    if csv_path.exists():
        return csv_path

    # 2. Extract Training/Testing and filename
    parts = csv_path.parts

    split_name = None

    if "Training" in parts:
        split_name = "Training"
    elif "Testing" in parts:
        split_name = "Testing"

    if split_name is not None:
        split_index = parts.index(split_name)

        # Example:
        # Training / notumor / Tr-no_626.jpg
        relative_path = Path(*parts[split_index:])

        candidate = RAW_DIR / relative_path

        if candidate.exists():
            return candidate

    # 3. Last-resort filename search
    filename = csv_path.name

    matches = list(RAW_DIR.rglob(filename))

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple images found for filename: {filename}"
        )

    raise FileNotFoundError(
        f"Image could not be resolved: {csv_path}"
    )


print("Path resolver ready!")

Path resolver ready!


In [30]:
# ============================================
# Test Path Resolver
# ============================================

test_path = train_df.iloc[0]["image_path"]

print("CSV path:")
print(test_path)

resolved_path = resolve_image_path(test_path)

print("\nResolved path:")
print(resolved_path)

print("\nExists:", resolved_path.exists())

CSV path:
/content/MediScan/Training/notumor/Tr-no_626.jpg

Resolved path:
/content/drive/MyDrive/MediScan/data/raw/Training/notumor/Tr-no_626.jpg

Exists: True


In [31]:
# ============================================
# Day 6 - Final Dataset Class
# ============================================

class BrainMRIDataset(Dataset):

    def __init__(self, dataframe, transform=None, class_to_idx=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        # Resolve old CSV path to current Drive path
        image_path = resolve_image_path(row["image_path"])

        # Convert class name to integer
        class_name = str(row["label"])
        label = self.class_to_idx[class_name]

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Apply transform
        if self.transform:
            image = self.transform(image)

        return image, label


print("Final BrainMRIDataset ready!")

Final BrainMRIDataset ready!


In [32]:
# ============================================
# Recreate Dataset Objects
# ============================================

train_dataset = BrainMRIDataset(
    train_df,
    transform=train_augmentation,
    class_to_idx=class_to_idx
)

val_dataset = BrainMRIDataset(
    val_df,
    transform=evaluation_transform,
    class_to_idx=class_to_idx
)

test_dataset = BrainMRIDataset(
    test_df,
    transform=evaluation_transform,
    class_to_idx=class_to_idx
)

print("Datasets recreated successfully!")

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Datasets recreated successfully!
Train: 5040
Validation: 1080
Test: 1080


In [33]:
# ============================================
# Day 6 - Dataset Verification
# ============================================

image, label = train_dataset[0]

print("Image shape :", image.shape)
print("Image dtype :", image.dtype)
print("Label       :", label)
print("Label type  :", type(label))
print("Pixel min   :", image.min().item())
print("Pixel max   :", image.max().item())

Image shape : torch.Size([3, 224, 224])
Image dtype : torch.float32
Label       : 2
Label type  : <class 'int'>
Pixel min   : -2.1179039478302
Pixel max   : 2.640000104904175


In [34]:
# ============================================
# Day 6 - Recreate DataLoaders
# ============================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

print("DataLoaders recreated successfully!")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders recreated successfully!
Train batches: 158
Validation batches: 34
Test batches: 34


In [35]:
# ============================================
# Day 6 - Loss Compatibility Test
# ============================================

model.eval()

with torch.no_grad():
    images, labels = next(iter(train_loader))

    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)

print("Input shape :", images.shape)
print("Output shape:", outputs.shape)
print("Labels shape:", labels.shape)
print("Labels dtype:", labels.dtype)
print("Loss value  :", loss.item())

Input shape : torch.Size([32, 3, 224, 224])
Output shape: torch.Size([32, 4])
Labels shape: torch.Size([32])
Labels dtype: torch.int64
Loss value  : 1.47794771194458


In [36]:
# ============================================
# DAY 6 FINAL VERIFICATION
# ============================================

print("========================================")
print("DAY 6 TRAINING LOOP VERIFICATION")
print("========================================")

print("Model              : EfficientNet-B0")
print("Device             :", device)
print("Loss Function      :", criterion.__class__.__name__)
print("Optimizer          :", optimizer.__class__.__name__)
print("Learning Rate      :", optimizer.param_groups[0]["lr"])
print("Scheduler           :", scheduler.__class__.__name__)
print("Scheduler Step Size:", scheduler.step_size)
print("Scheduler Gamma    :", scheduler.gamma)

print("\nBatch verification:")
print("Input shape        :", images.shape)
print("Output shape       :", outputs.shape)
print("Labels shape       :", labels.shape)
print("Labels dtype       :", labels.dtype)
print("Loss               :", loss.item())

print("\n========================================")
print("DAY 6 VERIFICATION PASSED")
print("========================================")

DAY 6 TRAINING LOOP VERIFICATION
Model              : EfficientNet-B0
Device             : cuda
Loss Function      : CrossEntropyLoss
Optimizer          : Adam
Learning Rate      : 0.001
Scheduler           : StepLR
Scheduler Step Size: 5
Scheduler Gamma    : 0.1

Batch verification:
Input shape        : torch.Size([32, 3, 224, 224])
Output shape       : torch.Size([32, 4])
Labels shape       : torch.Size([32])
Labels dtype       : torch.int64
Loss               : 1.47794771194458

DAY 6 VERIFICATION PASSED
